<a href="https://colab.research.google.com/github/AhmedCode110/AC-MOT/blob/main/notebooks/AC_MOT_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AC-MOT portable Colab runner — latest version
GitHub is the source of truth. Colab is only the runner.

Every stage below explains what is running, why it is needed, elapsed time, and what remains. Known-size work in v14 also prints a progress bar and ETA. Network/package work reports real elapsed time instead of inventing an ETA.


## Stage 1/6 — Mount Google Drive
This mounts Drive so the runner can read VisDrone and save persistent results. Realtime frame timing later uses local Colab SSD, not Drive.


In [ ]:
import time
NOTEBOOK_START = time.perf_counter()
STAGE_TIMES = {}

def fmt_time(seconds):
    seconds=max(float(seconds),0.0)
    if seconds < 60: return f'{seconds:.1f}s'
    m,s=divmod(int(round(seconds)),60); return f'{m}m {s:02d}s'

def begin_stage(number,title,now,why,remaining):
    print('\n'+'='*92,flush=True)
    print(f'[{number}/6] {title}',flush=True)
    print(f'[NOW] {now}',flush=True)
    print(f'[WHY] {why}',flush=True)
    print(f'[REMAINING AFTER THIS] {remaining}',flush=True)
    return time.perf_counter()

def end_stage(number,started):
    elapsed=time.perf_counter()-started; STAGE_TIMES[number]=elapsed
    print(f'[DONE {number}/6] elapsed={fmt_time(elapsed)} | notebook_total={fmt_time(time.perf_counter()-NOTEBOOK_START)}',flush=True)

t=begin_stage(1,'Mount Google Drive','Connecting /content/drive to Google Drive.','Dataset/results are on Drive.','GitHub sync -> dependencies -> config -> preflight -> AC-MOT run')
from google.colab import drive
drive.mount('/content/drive')
end_stage(1,t)


## Stage 2/6 — Authenticate and sync GitHub
Reads `GITHUB_TOKEN` from Colab Secrets, validates it, verifies private-repo access, then clones/pulls `main`. The token is never printed or stored in the notebook/Git URL/Git config. The exact commit is printed before anything is executed.


In [ ]:
t=begin_stage(2,'Sync latest AC-MOT from GitHub','Validating token, repo access, then clone/pull main.','Prevents Colab from running stale research code.','dependencies -> config -> preflight -> AC-MOT run')
from pathlib import Path
import json, os, sys, subprocess, tempfile, urllib.request, urllib.error
from google.colab import userdata
REPO_URL='https://github.com/AhmedCode110/AC-MOT.git'; REPO_API='https://api.github.com/repos/AhmedCode110/AC-MOT'; USER_API='https://api.github.com/user'; REPO=Path('/content/AC-MOT')
try: token=userdata.get('GITHUB_TOKEN')
except Exception as exc: raise RuntimeError('Colab Secret GITHUB_TOKEN is missing or Notebook access is disabled.') from exc
if not token: raise RuntimeError('Colab Secret GITHUB_TOKEN is empty.')

def github_json(url):
    request=urllib.request.Request(url,headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28','User-Agent':'AC-MOT-Colab'})
    try:
        with urllib.request.urlopen(request,timeout=20) as response: return response.status,json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as exc:
        body=exc.read().decode('utf-8',errors='replace')
        try: detail=json.loads(body).get('message',body)
        except Exception: detail=body
        return exc.code,{'message':detail}

sub=time.perf_counter(); print('[2.1/2] [NOW] Validating GITHUB_TOKEN...',flush=True)
user_status,user_data=github_json(USER_API)
if user_status!=200: raise RuntimeError(f'GITHUB_TOKEN authentication failed HTTP {user_status}: {user_data.get("message")}')
github_user=user_data.get('login'); print(f'[OK] token valid | user={github_user} | elapsed={fmt_time(time.perf_counter()-sub)}',flush=True)
sub=time.perf_counter(); print('[2.2/2] [NOW] Verifying AhmedCode110/AC-MOT access...',flush=True)
repo_status,repo_data=github_json(REPO_API)
if repo_status!=200: raise RuntimeError(f'Token cannot read AC-MOT HTTP {repo_status}: {repo_data.get("message")}')
print(f'[OK] private repo accessible | elapsed={fmt_time(time.perf_counter()-sub)}',flush=True)

sub=time.perf_counter()
with tempfile.TemporaryDirectory() as tmp:
    env=os.environ.copy(); env['GIT_TERMINAL_PROMPT']='0'; env['ACMOT_GH_TOKEN']=token; env['ACMOT_GH_USER']=github_user
    helper=Path(tmp)/'askpass'; helper.write_text('#!/usr/bin/env python3\nimport os, sys\nprompt=sys.argv[1] if len(sys.argv)>1 else \"\"\nprint(os.environ[\"ACMOT_GH_USER\"] if \"Username\" in prompt else os.environ[\"ACMOT_GH_TOKEN\"])\n'); helper.chmod(0o700); env['GIT_ASKPASS']=str(helper)
    command=['git','-c','credential.helper=']
    if (REPO/'.git').is_dir():
        print('[GITHUB] Repository exists -> pull --ff-only origin main',flush=True); result=subprocess.run(command+['-C',str(REPO),'pull','--ff-only','origin','main'],env=env,text=True,capture_output=True)
    else:
        print('[GITHUB] Repository absent -> clone main',flush=True); result=subprocess.run(command+['clone','--branch','main',REPO_URL,str(REPO)],env=env,text=True,capture_output=True)
    if result.stdout.strip(): print(result.stdout,flush=True)
    if result.stderr.strip(): print(result.stderr,flush=True)
    result.check_returncode()
commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
print(f'[OK] Repository ready | commit={commit} | sync_elapsed={fmt_time(time.perf_counter()-sub)}',flush=True)
end_stage(2,t)


## Stage 3/6 — Install pinned dependencies and TrackEval
Installs `requirements.txt` and prepares the pinned TrackEval revision. Network/package operations do not expose a trustworthy remaining-work total, so the cell shows real elapsed time instead of a fake ETA.


In [ ]:
t=begin_stage(3,'Install pinned dependencies and TrackEval','Installing requirements then TrackEval.','Package drift can change FPS and metrics.','config -> preflight -> AC-MOT run')
sub=time.perf_counter(); print('[3.1/2] [NOW] pip install requirements | ETA=not reliably knowable',flush=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
print(f'[OK] requirements installed | elapsed={fmt_time(time.perf_counter()-sub)}',flush=True)
sub=time.perf_counter(); print('[3.2/2] [NOW] preparing pinned TrackEval | ETA=not reliably knowable',flush=True)
subprocess.run([sys.executable,str(REPO/'scripts/setup_trackeval.py'),'/content/TrackEval'],check=True)
print(f'[OK] TrackEval ready | elapsed={fmt_time(time.perf_counter()-sub)}',flush=True)
end_stage(3,t)


## Stage 4/6 — Load the active version/config
Reads `configs/active_config.txt`, validates dataset/output paths, and prints the exact AC-MOT version, speed-test script, backend, FPS target and gate before running anything expensive.


In [ ]:
t=begin_stage(4,'Load active AC-MOT version/config','Reading active_config.txt and validating Drive paths.','Prevents running the wrong version or result path.','preflight -> AC-MOT run')
import uuid
CONFIG_NAME=(REPO/'configs'/'active_config.txt').read_text().strip(); CONFIG_FILE=REPO/'configs'/CONFIG_NAME; CFG=json.loads(CONFIG_FILE.read_text())
print(f'[CONFIG] file={CONFIG_NAME}',flush=True); print(f'[CONFIG] version={CFG.get("version","legacy/unversioned")}',flush=True); print(f'[CONFIG] mode={CFG["mode"]}',flush=True)
print(f'[CONFIG] speedtest_script={CFG.get("speedtest_script","scripts/speedtest_top3.py")}',flush=True); print(f'[CONFIG] backend={CFG.get("backend","default")}',flush=True); print(f'[CONFIG] target_fps={CFG.get("target_fps")} gate_frames={CFG.get("gate_frames")}',flush=True)
DATASET_ON_DRIVE=Path(CFG['dataset']); assert (DATASET_ON_DRIVE/'annotations').is_dir(); assert (DATASET_ON_DRIVE/'sequences').is_dir(); print('[OK] Dataset paths valid.',flush=True)
RESULTS=Path(CFG['output_root']); RESULTS.mkdir(parents=True,exist_ok=True); probe=RESULTS/('.write_probe_'+uuid.uuid4().hex); probe.write_text('probe'); probe.unlink(); print(f'[OK] Results folder writable: {RESULTS}',flush=True)
CONFIG_PATH=Path('/content')/('acmot_config_'+uuid.uuid4().hex+'.json'); CONFIG_PATH.write_text(json.dumps(CFG,indent=2)); print(f'[OK] Runtime config ready: {CONFIG_PATH}',flush=True)
end_stage(4,t)


## Stage 5/6 — Environment preflight
Checks package pins, CUDA and Tesla T4 before expensive work. v14 also prints `ACTIVE VERSION: v14` and the selected v14 speed-test script.


In [ ]:
t=begin_stage(5,'Environment preflight','Checking package pins, CUDA, T4, active version and script.','Failing here is cheaper than failing mid-run.','AC-MOT run')
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH),'--check'],check=True)
end_stage(5,t)


## Stage 6/6 — Run AC-MOT
For v14 the child runner prints backend build/load status, copy/calibration/warmup progress + ETA, sequence/frame progress, cumulative/recent FPS, requested/used resolution, SCI/scene, per-stage latency, the 300-frame realtime gate, and the final Drive result path.


In [ ]:
t=begin_stage(6,'Run active AC-MOT version','Starting scripts/run.py; v14 prints detailed progress/ETA/profile.','This is the actual experiment; every run saves to a new Drive envelope.','none — final stage')
print(f'[RUN] version={CFG.get("version")} script={CFG.get("speedtest_script")} target={CFG.get("target_fps")} FPS',flush=True)
subprocess.run([sys.executable,str(REPO/'scripts/run.py'),'--config',str(CONFIG_PATH)],check=True)
end_stage(6,t)
print('\n'+'='*92,flush=True); print('[OK] ALL NOTEBOOK STAGES COMPLETED',flush=True); print(f'[TOTAL] elapsed={fmt_time(time.perf_counter()-NOTEBOOK_START)}',flush=True); print(f'[RESULTS ROOT] {CFG["output_root"]}',flush=True); print('[STAGE TIMES]',{k:fmt_time(v) for k,v in STAGE_TIMES.items()},flush=True)


## Daily workflow
Edit/push in VS Code -> open this notebook from GitHub -> choose Tesla T4 -> Run all -> confirm Stage 4 prints the expected version -> use `speedtest_results.json` as the source of truth for PASS/FAIL and stage latency.
